# Notebook 3: Machine Learning
Our objective is to train a robust Random Forest Classifier capable of accurately predicting whether a given molecule will bind to the Androgen Receptor.

A major focus of this notebook is conducting a head-to-head performance comparison between two industry-standard structural representations:
* MACCS Keys (166-bit): A pre-defined, rigid dictionary of common structural fragments.
* Morgan Fingerprints (2048-bit, Radius 3): A high-dimensional, circular topological fingerprint that captures complex, localized atomic environments.

We will evaluate both approaches using rigorous cross-validation and standard classification metrics (ROC-AUC, Precision, Recall), and the superior model will be saved for deployment in downstream virtual screening.

In [ ]:
from pathlib import Path
from warnings import filterwarnings
import time

import pandas as pd
import numpy as np
import seaborn as sns
from sklearn import svm, metrics, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, train_test_split, StratifiedKFold
from sklearn.metrics import auc, accuracy_score, recall_score
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator

# Silence some expected warnings
filterwarnings("ignore")
# Fix seed for reproducible results
SEED = 22
np.random.seed(SEED)

In [ ]:
HERE = Path.cwd()
DATA = HERE / "data"

### 1. Data Loading and Target Definition

Before we can train our machine learning algorithms, we need to load our curated dataset from Notebook 2 and define the "target" variable (`y`) that our model will try to predict.

Because we are building a Classification model (predicting a category rather than an exact number), we need to convert our continuous bioactivity scores into binary labels. We will use a strict biological threshold to separate the highly potent drugs from the weak ones:
* **Active (1.0):** Compounds with a pIC50 >= 6.3
* **Inactive (0.0):** Compounds with a pIC50 < 6.3

In [ ]:
# Read data from Notebook 2
filtered_df = pd.read_csv(
    DATA / "AR_filtered.csv",
    index_col=0,
)
print("Shape of dataframe : ", filtered_df.shape)

In [ ]:
filtered_df.head()

In [ ]:
# Add column for activity
filtered_df["active"] = np.zeros(len(filtered_df))

# Mark every molecule as active (1.0) with an pIC50 of >= 6.3, 0 otherwise
filtered_df.loc[filtered_df[filtered_df.pIC50 >= 6.3].index, "active"] = 1.0

print("Number of active compounds:", int(filtered_df.active.sum()))
print("Number of inactive compounds:", len(filtered_df) - int(filtered_df.active.sum()))

In [ ]:
filtered_df.head()

### 2. Feature Engineering
Machine learning algorithms cannot directly read raw text like SMILES strings; they require strict numerical inputs. In this section, we will translate our chemical structures into continuous mathematical arrays that our models can learn from.

We will execute this pipeline in three distinct steps:

1. **Boolean Cleanup:** We will convert our logical columns (`active`) from `True/False` into binary integers (`1`/`0`) to ensure compatibility with `scikit-learn`.
2. **Structural Fingerprinting:** We will generate two completely different sets of chemical fingerprints to set up an A/B test for our model:
    * MACCS Keys: A 167-bit array that asks predefined "yes/no" questions about the presence of specific chemical fragments.
    * Morgan Fingerprints (Radius 3): A highly detailed 2048-bit array that maps the circular neighborhood of atoms surrounding every single atom in the molecule.
3. **Feature Integration:** To give our models the most comprehensive view of each molecule, we will concatenate the physical properties (our 5 Lipinski descriptors) with the structural properties (the fingerprints) into a single, unified 1D feature vector ($X$) for every compound.

In [ ]:
# Create an independent copy of the data to keep our original dataframe pristine
compound_df = filtered_df.copy()

In [ ]:
# Data formatting for machine learning
# Convert boolean columns into binary integers as Scikit-learn algorithms require strictly numerical data
compound_df['active'] = compound_df['active'].astype(int)

In [ ]:
# Define a function to convert SMILES strings into mathematical fingerprint array for machine learning
def smiles_to_fp(smiles, method="maccs", n_bits=2048):

    # Convert smiles to RDKit mol object
    mol = Chem.MolFromSmiles(smiles)

    # Option 1: Convert RDKit mol object to MACCS keys
    if method == "maccs":
        return np.array(MACCSkeys.GenMACCSKeys(mol))

    # Option 2: Convert RDKit mol object to Morgan fingerprints
    if method == "morgan3":
        fpg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=n_bits)
        return np.array(fpg.GetFingerprint(mol))
    else:
        # Fallback: Catch typos in the method argument and default safetly to MACCS
        print(f"Warning: Wrong method specified: {method}. Default will be used instead.")
        return np.array(MACCSkeys.GenMACCSKeys(mol))

In [ ]:
# Fingerprint generation
# Convert the SMILES strings into two different representation
compound_df['maccs_fp'] = compound_df['smiles'].apply(smiles_to_fp, method="maccs")
compound_df['morgan_fp'] = compound_df['smiles'].apply(smiles_to_fp, method="morgan3")
print("Fingerprint generation complete!")

In [ ]:
# Feature integration
# Define the Lipinski's Rule of 5 properties we calculated in Notebook 2
lipinski_cols = ["molecular_weight", "n_hba", "n_hbd", "logp"]

# Define a function to combine the Lipinski features with the fingerprints
def combine_features(row, fp_col):

    # Extract Lipinski features as a float array
    lipinski = row[lipinski_cols].values.astype(float)

    # Extract the fingerprint array
    fp = np.array(row[fp_col])

    # Stitch them together
    return np.concatenate((lipinski, fp))

# Apply the combination function to create our final ML-ready columns
compound_df['maccs_combined'] = compound_df.apply(lambda row: combine_features(row, 'maccs_fp'), axis=1)
compound_df['morgan_combined'] = compound_df.apply(lambda row: combine_features(row, 'morgan_fp'), axis=1)

print(f"MACCS Feature Vector Length: {len(compound_df['maccs_combined'].iloc[0])}")
print(f"Morgan Feature Vector Length: {len(compound_df['morgan_combined'].iloc[0])}")

In [ ]:
# Check the data frame
compound_df.head()

### 3. Data Splitting (Train/Test)
Before training our machine learning models, we must split our dataset into a **Training Set** (80%) and a **Testing Set** (20%). The testing set acts as "unseen chemistry"—data the model has never encountered—allowing us to evaluate its true predictive power.

In [ ]:
# Lock the random seed to ensure that our split is 100% reproducible
SEED = 42

# Extract the target variable
y = compound_df['active'].tolist()

# Split 1: Lipinski + MACCS
# 20% of the data is to be used to test the model's performance on unseen data
x_maccs = compound_df['maccs_combined'].tolist()
train_x_maccs, test_x_maccs, train_y_maccs, test_y_maccs = train_test_split(
    x_maccs, y, test_size=0.2, random_state=SEED
)

# Split 2: Lipinski + Morgan
# We use the same random_state (seed=42) so the Morgan model gets tested on the exact same 20% of molecules as the MACCS model
x_morgan = compound_df['morgan_combined'].tolist()
train_x_morgan, test_x_morgan, train_y_morgan, test_y_morgan = train_test_split(
    x_morgan, y, test_size=0.2, random_state=SEED
)

# Validate the shapes of the arrays
print("\nData successfully split!")
print(f"MACCS Feature Array Length: {len(train_x_maccs[0])} (5 Lipinski + 167 MACCS bits)")
print(f"Morgan Feature Array Length: {len(train_x_morgan[0])} (5 Lipinski + 2048 Morgan bits)")

### 4. Machine Learning Pipeline (Helper Functions)
We first define a robust toolkit of evaluation functions.
By standardizing how we measure success, we ensure that both our MACCS Keys and Morgan Fingerprint models are judged on the exact same criteria without any data leakage.

Our toolkit consists of four distinct components:
1. **Core Metrics (`model_performance`):** A standardized function that calculates Accuracy, Sensitivity, Specificity, and the Area Under the Curve (AUC).
2. **Standard Validation (`model_training_and_validation`):** A function that handles the fitting of a model on a training set and immediately evaluates it on a held-out test set.
3. **Robustness Testing (`crossvalidation`):** This function executes a Stratified K-Fold cross-validation. It intelligently splits the training data into 5 folds while protecting the active/inactive class ratios. Crucially, it uses `clone()` to create a fresh, untrained model for every fold, ensuring no memory leakage. We track the standard deviation of the scores to prove model stability.
4. **Visual Evaluation (`plot_roc_curves_for_models`):** Our ultimate visualization tool. It takes our fully trained models, tests them against the locked-away test data, and plots them head-to-head on a single Receiver Operating Characteristic (ROC) curve to visually declare a champion.

In [ ]:
# Helper function 1
def model_performance(ml_model, test_x, test_y, verbose=True):
    """
    Helper function to calculate model performance

    Parameters
    ----------
    ml_model: sklearn model object
        The machine learning model to train.
    test_x: list
        Molecular fingerprints for test set.
    test_y: list
        Associated activity labels for test set.
    verbose: bool
        Print performance measure (default = True)

    Returns
    -------
    tuple:
        Accuracy, sensitivity, specificity, auc on test set.
    """

    # Prediction probability on test set
    test_prob = ml_model.predict_proba(test_x)[:, 1]

    # Prediction class on test set
    test_pred = ml_model.predict(test_x)

    # Performance of model on test set
    accuracy = accuracy_score(test_y, test_pred)
    sens = recall_score(test_y, test_pred)
    spec = recall_score(test_y, test_pred, pos_label=0)
    auc = roc_auc_score(test_y, test_prob)

    if verbose:
        # Print performance results
        print(f"Accuracy: {accuracy:.2}")
        print(f"Sensitivity: {sens:.2f}")
        print(f"Specificity: {spec:.2f}")
        print(f"AUC: {auc:.2f}")

    return accuracy, sens, spec, auc

In [ ]:
# Helper function 2
def model_training_and_validation(ml_model, name, splits, verbose=True):
    """
    Fit a machine learning model on a random train-test split of the data and return the performance measures.

    Parameters
    ----------
    ml_model: sklearn model object
        The machine learning model to train.
    name: str
        Name of machine learning algorithm: RF, SVM, ANN
    splits: list
        List of desciptor and label data: train_x, test_x, train_y, test_y.
    verbose: bool
        Print performance info (default = True)

    Returns
    -------
    tuple:
        Accuracy, sensitivity, specificity, auc on test set.

    """
    train_x, test_x, train_y, test_y = splits

    # Fit the model
    ml_model.fit(train_x, train_y)

    # Calculate model performance results
    accuracy, sens, spec, auc = model_performance(ml_model, test_x, test_y, verbose)

    return accuracy, sens, spec, auc

In [ ]:
def crossvalidation(ml_model, x, y, n_folds=5, verbose=True):
    """
    Evaluates a machine learning model's robustness using Stratified K-Fold cross-validation.

    Parameters
    ----------
    ml_model: sklearn model object
        The machine learning model to train.
    x: list or np.ndarray
        The feature matrix (e.g., MACCS or Morgan arrays).
    y: list or np.ndarray
        The binary target labels (1 for active, 0 for inactive).
    n_folds: int, optional
        Number of folds for cross-validation.
    verbose: bool, optional
        Performance measures are printed.

    Returns
    -------
    tuple
        Lists of the metrics for each fold.
    """
    t0 = time.time()

    # StratifiedKFold protects the ratio of Actives to Inactives across all folds
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    acc_per_fold = []
    sens_per_fold = []
    spec_per_fold = []
    auc_per_fold = []

    # Convert incoming data to numpy arrays so we can easily slice them by index
    x_array = np.array(x)
    y_array = np.array(y)

    # Loop over the folds
    for train_index, test_index in kf.split(x_array, y_array):

        # Clone model: we want a fresh, blank brain for every fold!
        fold_model = clone(ml_model)

        # Slice the data into training and testing chunks for this specific fold
        x_train_fold, x_test_fold = x_array[train_index], x_array[test_index]
        y_train_fold, y_test_fold = y_array[train_index], y_array[test_index]

        # Fit the model strictly on the training chunk
        fold_model.fit(x_train_fold, y_train_fold)

        # Calculate performance using your helper function
        # (We set verbose=False here so it doesn't print 5 times in a row)
        accuracy, sens, spec, auc = model_performance(fold_model, x_test_fold, y_test_fold, verbose=False)

        # Save results
        acc_per_fold.append(accuracy)
        sens_per_fold.append(sens)
        spec_per_fold.append(spec)
        auc_per_fold.append(auc)

    # Print statistics of results
    if verbose:
        print(
            f"Mean accuracy:    {np.mean(acc_per_fold):.3f} \t (± {np.std(acc_per_fold):.3f})\n"
            f"Mean sensitivity: {np.mean(sens_per_fold):.3f} \t (± {np.std(sens_per_fold):.3f})\n"
            f"Mean specificity: {np.mean(spec_per_fold):.3f} \t (± {np.std(spec_per_fold):.3f})\n"
            f"Mean AUC:         {np.mean(auc_per_fold):.3f} \t (± {np.std(auc_per_fold):.3f})\n"
            f"Time taken:       {time.time() - t0:.2f}s\n"
        )

    return acc_per_fold, sens_per_fold, spec_per_fold, auc_per_fold

In [ ]:
def plot_roc_curves_for_models(models, save_png=False):
    """
    Helper function to plot customized roc curve.

    Parameters
    ----------
    models: dict
        Dictionary of pretrained machine learning models.
    test_x: list
        Molecular fingerprints for test set.
    test_y: list
        Associated activity labels for test set.
    save_png: bool
        Save image to disk (default = False)

    Returns
    -------
    fig:
        Figure.
    """

    fig, ax = plt.subplots()

    # Below for loop iterates through your models list
    for model in models:
        # Select the model
        ml_model = model["model"]
        test_x = model["test_x"]
        test_y = model["test_y"]

        # Prediction probability on test set
        test_prob = ml_model.predict_proba(test_x)[:, 1]
        # Prediction class on test set
        test_pred = ml_model.predict(test_x)
        # Compute False postive rate and True positive rate
        fpr, tpr, thresholds = metrics.roc_curve(test_y, test_prob)
        # Calculate Area under the curve to display on the plot
        auc = roc_auc_score(test_y, test_prob)
        # Plot the computed values
        ax.plot(fpr, tpr, label=(f"{model['label']} AUC area = {auc:.2f}"))

    # Custom settings for the plot
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Receiver Operating Characteristic (ROC)")
    ax.legend(loc="lower right")
    # Save plot
    if save_png:
        fig.savefig(f"{DATA}/roc_auc", dpi=300, bbox_inches="tight", transparent=True)
    return fig

### 5. Random Forest Classifier Evaluation
We can first initialize our machine learning algorithm. We have chosen a **Random Forest Classifier**, an ensemble learning method that builds hundreds of decision trees and aggregates their predictions.

To ensure our model evaluation is mathematically rigorous and free from data leakage, we follow this order of operations:
1. **Initialization:** Initialize blank models with a locked random seed for reproducibility.
2. **Cross-Validation:** Cross-validate the model using only the training set. This allows us to evaluate model stability and compare the MACCS vs. Morgan fingerprints without leaking the test data.
3. **Final Evaluation:** After proving our models' stability via cross-validation, we train our algorithms on 100% of the training set and test on the 20% test set.
4. **Plotting ROC Curve:** Finally, we generate the performance metrics and plot the Receiver Operating Characteristic (ROC) curves. This visualizes the trade-off between the True Positive Rate (Sensitivity) and the False Positive Rate (1 - Specificity). The Area Under the Curve (AUC) serves as our definitive score.

In [ ]:
# Bundle the test/train arrays into clean lists
maccs_splits = [train_x_maccs, test_x_maccs, train_y_maccs, test_y_maccs]
morgan_splits = [train_x_morgan, test_x_morgan, train_y_morgan, test_y_morgan]

# Define the parameters for the Random Forest model
param = {
    "n_estimators": 100,  # number of trees to grows
    "criterion": "entropy",  # cost function to be optimized for a split
    "random_state": 42  # Lock the seed
}

# Initialize two separate models
model_RF_maccs = RandomForestClassifier(**param)
model_RF_morgan = RandomForestClassifier(**param)

In [ ]:
# Cross Validation of MAACS model (only on training set)
cv_maccs = crossvalidation(model_RF_maccs, train_x_maccs, train_y_maccs)

In [ ]:
# Cross Validation of Morgan model (only on training set)
cv_morgan = crossvalidation(model_RF_morgan, train_x_morgan, train_y_morgan)

In [ ]:
# Fit model on MACCS split (train on training set and validate with test set)
perf_RF_maccs = model_training_and_validation(model_RF_maccs, "RF", maccs_splits)

In [ ]:
# Fit model on Morgan split (train on training set and validate with test set)
perf_RF_morgan = model_training_and_validation(model_RF_morgan, "RF", morgan_splits)

In [ ]:
# Plot ROC curves for MACCS and Morgan model
# Package the trained models with their test sets
trained_models = [
    {
        "label": "RF + MACCS Keys",
        "model": model_RF_maccs,
        "test_x": test_x_maccs,
        "test_y": test_y_maccs
    },
    {
        "label": "RF + Morgan FP",
        "model": model_RF_morgan,
        "test_x": test_x_morgan,
        "test_y": test_y_morgan
    }
]

# Generate the ROC curves
plot_roc_curves_for_models(trained_models)

The evaluation metrics indicate that **Morgan fingerprints** yield a highly accurate and more robust predictive model.

In [ ]:
# Save the Morgan model to a file
import joblib
joblib.dump(model_RF_morgan, 'morgan_model.pkl')
print("Champion model securely saved and ready for deployment!")

### 6. Feature Importance
Random Forests provide a built-in metric called **Mean Decrease in Impurity (MDI)**. Every time the algorithm splits the data using a specific chemical property, it calculates how much that split successfully separated the Active drugs from the Inactive ones.

Because we engineered our features by combining **Lipinski rules** with **Morgan fingerprints**, we can extract these MDI scores to see exactly which combination of physics and structure drove the RF model's success. Below, we map these scores to their real chemical names and visualize the Top 20 most critical features.

In [ ]:
# Machine learning feature importance ranking
# Recreate the 5 Lipinski names we used earlier
lipinski_names = ["Molecular_Weight", "Num_HBA", "Num_HBD", "LogP"]

# Generate names for the 2048 Morgan bits (e.g., "Morgan_Bit_0", "Morgan_Bit_1")
morgan_names = [f"Morgan_Bit_{i}" for i in range(2048)]

# Combine them to perfectly match the 2,053 length of our train_x_morgan array
all_feature_names = lipinski_names + morgan_names

# Extract the built-in importance scores from our trained Morgan model
importances = model_RF_morgan.feature_importances_

# Bundle the names and scores into a clean Pandas DataFrame
feat_imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
})

# Sort the dataframe from highest importance to lowest, and slice out the Top 20
top_20_features = feat_imp_df.sort_values(by='Importance', ascending=False).head(20)

# Visualization
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=top_20_features, palette='viridis')

# Formatting for a professional look
plt.title('Top 20 Most Important Features (Morgan FP + Lipinski)', fontsize=14, pad=15)
plt.xlabel('Importance Score (Mean Decrease in Impurity)', fontsize=12)
plt.ylabel( 'Features', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()